# Monitor Pack 1 download progress

This notebook parses the live download log for `download-pack-hftransfer.py`, computes throughput, estimates completion time, and validates the token-vault layout. All cells are read-only: they do not modify the running download.

In [ ]:
import json, re, os, pathlib, datetime, sys
from collections import defaultdict

LOG_PATH = pathlib.Path(r'C:\R\LeafOS0.2.2\ProjectLeaf\install\windows\pack1_download.log')
VAULT_PATH = pathlib.Path.home() / '.cache' / 'huggingface' / 'flower_token_vault.dat'

print('LOG_PATH :', LOG_PATH)
print('VAULT_PATH:', VAULT_PATH)

## 1. Validate the token vault

The encrypted vault should exist and be non-empty. The actual decryption is left to `token_vault.exe` so the password is never handled in the notebook.

In [ ]:
def check_vault(vault: pathlib.Path):
    if not vault.exists():
        return {'status': 'missing', 'size': 0}
    sz = vault.stat().st_size
    return {'status': 'present', 'size': sz, 'readable': os.access(vault, os.R_OK)}

vault_info = check_vault(VAULT_PATH)
print(json.dumps(vault_info, indent=2))

## 2. Parse progress lines from a huggingface_hub log

The downloader emits ANSI progress bars like:

```
Qwen3.6-27B-Fable-Fus-711-UnHeretic-NM-D(...): downloading bytes: | 412MB, 2.55MB/s
```

We strip ANSI escapes and extract per-file megabytes and instantaneous speed.

In [ ]:
ANSI_RE = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')
PROGRESS_RE = re.compile(r
    r'downloading bytes:\s*'
    r'.*?'
    r'(?P<mb>[0-9]+(?:\.[0-9]+)?)\s*MB'
    r'(?:\s*,\s*(?P<mbs>[0-9]+(?:\.[0-9]+)?)\s*MB/s)?'
)

def parse_log(path: pathlib.Path):
    if not path.exists():
        print('Log not found:', path)
        return []
    records = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for lineno, raw in enumerate(f, 1):
            line = ANSI_RE.sub('', raw)
            if 'downloading bytes:' not in line:
                continue
            m = PROGRESS_RE.search(line)
            if not m:
                continue
            mb = float(m.group('mb'))
            mbs = float(m.group('mbs')) if m.group('mbs') else None
            records.append({'line': lineno, 'mb': mb, 'mbs': mbs, 'raw': line.strip()})
    return records

records = parse_log(LOG_PATH)
print(f'Parsed {len(records)} progress samples')
print(json.dumps(records[-3:], indent=2))

## 3. Aggregate throughput and ETA

Assuming the registry header printed a total size, we derive an overall ETA from the latest samples.

In [ ]:
TOTAL_GIB = 193.27  # from registry summary
TOTAL_MIB = TOTAL_GIB * 1024

if records:
    latest = records[-1]['mb']
    speeds = [r['mbs'] for r in records if r['mbs']]
    avg_speed = sum(speeds) / len(speeds) if speeds else 0.0
    recent_speeds = [r['mbs'] for r in records[-100:] if r['mbs']]
    recent_avg = sum(recent_speeds) / len(recent_speeds) if recent_speeds else 0.0
    remaining = TOTAL_MIB - latest
    eta_sec = remaining / recent_avg if recent_avg else float('inf')
    eta = datetime.timedelta(seconds=int(eta_sec))
    pct = 100.0 * latest / TOTAL_MIB

    print(f'Total pack:     {TOTAL_GIB:.2f} GiB')
    print(f'Latest sample:  {latest:.2f} MiB')
    print(f'Progress:       {pct:.3f}%')
    print(f'Average speed:  {avg_speed:.2f} MiB/s')
    print(f'Recent speed:   {recent_avg:.2f} MiB/s')
    print(f'ETA (recent):   {eta}')
else:
    print('No records to summarise.')

## 4. Optional: plot bytes over time

If `matplotlib` is installed, render a simple progress curve.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('matplotlib not installed; skipping plot')
    raise SystemExit

if records:
    xs = list(range(len(records)))
    ys = [r['mb'] for r in records]
    plt.figure(figsize=(10, 4))
    plt.plot(xs, ys, label='Downloaded MiB')
    plt.axhline(TOTAL_MIB, color='r', linestyle='--', label=f'Target {TOTAL_GIB} GiB')
    plt.xlabel('Sample index')
    plt.ylabel('MiB')
    plt.title('Pack 1 download progress')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## 5. Restart behavior

`hf_hub_download` resumes whenever possible, so interrupting and rerunning `_start_pack1_download.ps1` is safe. Always invoke it through `pwsh`, not Windows PowerShell 5.1.